# 4단계 실험: 초안 → SEO 메타 (OpenAI 구조화 출력)

3단계에서 생성된 마크다운 초안을 받아 SEO 친화적인 제목/메타디스크립션/태그/슬러그를 생성합니다.

**전제**: `OPENAI_API_KEY`. 초안은 즉석 생성하거나 임의 텍스트로 대체할 수 있습니다.

## 0. 환경 설정

In [1]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

for k in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY", "TAVILY_API_KEY"):
    print(f"{k}: {'OK' if os.getenv(k) else 'MISSING'}")

OPENAI_API_KEY: OK
ANTHROPIC_API_KEY: OK
TAVILY_API_KEY: OK


## 1. 초안 준비

옵션 A: 3단계 `write_draft()` 로 즉석 생성. 비용 절약하려면 옵션 B (임의 텍스트) 로 SEO 모듈만 단독 검증.

In [2]:
TOPIC = "외국인을 위한 한국 길거리 음식 추천"

# --- 옵션 A: 실제 초안 생성 ---
from llm_jacky.writing import write_draft
draft_result = write_draft(TOPIC, k=5)
DRAFT_TEXT = draft_result.draft

# --- 옵션 B: SEO 단독 검증용 더미 초안 (위 두 줄 주석 처리하고 사용) ---
# DRAFT_TEXT = (
#     "# 한국 길거리 음식 가이드\n\n"
#     "떡볶이는 매콤달콤한 떡 요리입니다. 호떡은 따뜻한 디저트입니다. "
#     "외국인 방문객은 명동, 광장시장에서 다양한 길거리 음식을 즐길 수 있습니다."
# )

print(DRAFT_TEXT[:400], "...")

# 한국 길거리 음식 완전 정복 🍢 외국인을 위한 필수 추천 가이드

## 들어가며

한국을 여행한다면 길거리 음식은 절대 빠뜨릴 수 없는 경험입니다. 한국에는 지역과 도시마다 다양한 길거리 음식이 있으며, 저렴한 가격에 현지 문화를 온몸으로 느낄 수 있는 최고의 방법이기도 합니다.[1] 지금부터 외국인 여행자라면 꼭 먹어봐야 할 한국 대표 길거리 음식들을 소개합니다.

---

## 🌶️ 1위 떡볶이 — 한국 길거리의 상징

한국 길거리 음식의 대표 주자는 단연 **떡볶이**입니다. 매콤하고 달콤한 소스에 쫀득한 떡과 어묵, 파, 양배추 등이 어우러진 음식으로, 어른과 아이 할 것 없이 남녀노소 모두 즐겨 먹습니다.[1] 특히 10대부터 30대 여성들에게 큰 인기를 끌고 있으며, 튀김·순대·오뎅국과 함께  ...


## 2. SEO 메타 생성

`generate_seo()` 는 Pydantic 구조화 출력을 사용해 항상 동일 스키마를 보장합니다.

In [3]:
from llm_jacky.seo import generate_seo

meta = generate_seo(TOPIC, DRAFT_TEXT)

print("title         :", meta.title, f"({len(meta.title)} chars)")
print("meta_desc     :", meta.meta_description, f"({len(meta.meta_description)} chars)")
print("tags          :", meta.tags)
print("slug          :", meta.slug)

title         : 외국인을 위한 한국 길거리 음식 추천 가이드 (24 chars)
meta_desc     : 한국 여행 시 꼭 맛봐야 할 길거리 음식, 떡볶이와 김밥을 포함한 다양한 추천 메뉴를 소개합니다. 한국의 맛과 문화를 경험해보세요! (73 chars)
tags          : ['한국 길거리 음식', '떡볶이', '김밥', '외국인 추천', '한식', '여행 가이드', '한국 음식']
slug          : korean-street-food-guide-for-foreigners


## 3. 길이/형식 sanity check

프롬프트가 실제로 제약을 잘 지키는지 확인합니다.

In [4]:
import re

checks = {
    "title <= 60": len(meta.title) <= 60,
    "meta_desc 130~170": 130 <= len(meta.meta_description) <= 170,
    "tags 5~8": 5 <= len(meta.tags) <= 8,
    "tags unique": len(meta.tags) == len(set(meta.tags)),
    "slug ascii-kebab": bool(re.fullmatch(r"[a-z0-9]+(-[a-z0-9]+)*", meta.slug)),
    "slug <= 50": len(meta.slug) <= 50,
}
for name, ok in checks.items():
    print(f"  {'PASS' if ok else 'WARN'}: {name}")

  PASS: title <= 60
  WARN: meta_desc 130~170
  PASS: tags 5~8
  PASS: tags unique
  PASS: slug ascii-kebab
  PASS: slug <= 50


## 4. 프롬프트 변형: 영어 SEO

예: 영문 권 독자 대상 메타 생성. 같은 스키마, 다른 톤.

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from llm_jacky.seo import SeoMeta

en_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an SEO editor for an English Korean-culture blog. Use only facts from the draft."),
    ("human", "Topic: {topic}\n\nDraft:\n{draft}\n\nGenerate English SEO metadata."),
])
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
en_meta = (en_prompt | llm.with_structured_output(SeoMeta)).invoke(
    {"topic": TOPIC, "draft": DRAFT_TEXT}
)
print("title :", en_meta.title)
print("desc  :", en_meta.meta_description)
print("tags  :", en_meta.tags)
print("slug  :", en_meta.slug)

title : Essential Guide to Korean Street Food for Foreigners
desc  : Discover the must-try Korean street foods for foreigners visiting Korea. From spicy tteokbokki to nutritious gimbap, explore the flavors of Korea's vibrant street food culture!
tags  : ['Korean street food', 'Tteokbokki', 'Gimbap', 'Korean cuisine', 'Travel Korea', 'Food guide', 'Cultural experience']
slug  : korean-street-food-guide


## 5. 발행 가능 형태로 합성 (마크다운 + 프론트매터)

초안 + SEO 메타 + 출처를 합쳐 한 파일로 떨어뜨립니다. 6단계 WordPress 발행 시 이 객체를 어댑터에 넘기면 됩니다.

In [6]:
from datetime import datetime

frontmatter = (
    "---\n"
    f"title: {meta.title!r}\n"
    f"description: {meta.meta_description!r}\n"
    f"slug: {meta.slug}\n"
    f"tags: {meta.tags}\n"
    f"date: {datetime.now().date().isoformat()}\n"
    "---\n\n"
)
post_md = frontmatter + DRAFT_TEXT

out_dir = PROJECT_ROOT / "data" / "drafts"
out_dir.mkdir(parents=True, exist_ok=True)
path = out_dir / f"{meta.slug or 'post'}.md"
path.write_text(post_md, encoding="utf-8")
print("saved:", path)
print("\n--- preview ---\n")
print(post_md[:600], "...")

saved: /Users/jackykim/project/llm-jacky/data/drafts/korean-street-food-guide-for-foreigners.md

--- preview ---

---
title: '외국인을 위한 한국 길거리 음식 추천 가이드'
description: '한국 여행 시 꼭 맛봐야 할 길거리 음식, 떡볶이와 김밥을 포함한 다양한 추천 메뉴를 소개합니다. 한국의 맛과 문화를 경험해보세요!'
slug: korean-street-food-guide-for-foreigners
tags: ['한국 길거리 음식', '떡볶이', '김밥', '외국인 추천', '한식', '여행 가이드', '한국 음식']
date: 2026-05-06
---

# 한국 길거리 음식 완전 정복 🍢 외국인을 위한 필수 추천 가이드

## 들어가며

한국을 여행한다면 길거리 음식은 절대 빠뜨릴 수 없는 경험입니다. 한국에는 지역과 도시마다 다양한 길거리 음식이 있으며, 저렴한 가격에 현지 문화를 온몸으로 느낄 수 있는 최고의 방법이기도 합니다.[1] 지금부터 외국인 여행자라면 꼭 먹어봐야 할 한국 대표 길거리 음식들을 소개합니다.

---

## 🌶️ 1위 떡볶이 — 한국 길거리의 상징

한국 길거리 음식의 대표 주자는 단연 **떡볶이**입니다. 매콤하고 달콤한 소스에 쫀득한 떡과 어묵, 파, 양배추 등이 어우러진 음식으로, 어른과 아이 할 것 없이 남녀노소 ...
